In [0]:

silver_path = "s3://my-datasets-for-demo/customer-churn-datasets/silver/transactions"

df_silver = spark.read.format("delta").load(silver_path)
df_silver.show()
df_silver.printSchema()

+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+---------------+--------------------+-------------+-------------+------------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|CustomerID|       Country|is_cancellation|is_negative_quantity|is_zero_price|is_adjustment|        LineAmount|
+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+---------------+--------------------+-------------+-------------+------------------+
| 489435|    22195|HEART MEASURING S...|      24|2009-12-01 07:46:00| 1.65|     13085|United Kingdom|          false|               false|        false|        false|39.599999999999994|
| 489437|    20971|PINK BLUE FELT CR...|      12|2009-12-01 09:08:00| 1.25|     15362|United Kingdom|          false|               false|        false|        false|              15.0|
| 489439|   16161P|   WRAP ENGLISH ROSE|      25|2009-12-01 09:28:00| 

In [0]:
from pyspark.sql.functions import col, max, datediff, lit, date_sub

# Drop NULL values
df_silver = df_silver.filter(
    col("CustomerID").isNotNull()
)

# Timeline setup
HOLDOUT_DAYS = 90
max_dataset_date = df_silver.select(max("InvoiceDate")).first()[0]
obs_df = df_silver.filter(col("InvoiceDate") <= date_sub(lit(max_dataset_date), HOLDOUT_DAYS))
holdout_df = df_silver.filter(col("InvoiceDate") > date_sub(lit(max_dataset_date), HOLDOUT_DAYS))

# Drop invalid values
df_purchases = obs_df.filter(
    (col("Quantity") > 0) &
    (col("Price") > 0) &
    (~col("is_cancellation")) &
    (~col("is_adjustment"))
)
holdout_purchases = holdout_df.filter(
    (col("Quantity") > 0) & 
    (col("Price") > 0) & 
    (~col("is_cancellation")) & 
    (~col("is_adjustment"))
)

In [0]:
print(df_silver.select(max("InvoiceDate")).first()[0])
print(obs_df.select(max("InvoiceDate")).first()[0])
print(df_purchases.select(max("InvoiceDate")).first()[0])

2011-12-09 12:50:00
2011-09-09 16:36:00
2011-09-09 15:53:00


In [0]:
from pyspark.sql.functions import col, sum, countDistinct, avg

invoice_features = (
    df_purchases
    .groupBy("CustomerID", "Invoice")
    .agg(
        sum("LineAmount").alias("order_value")
    )
)
customer_order_features = (
    invoice_features
    .groupBy("CustomerID")
    .agg(
        avg("order_value").alias("average_order_value"),
        sum("order_value").alias("total_spend"),
        countDistinct("Invoice").alias("purchase_frequency")
    )
)

In [0]:
from pyspark.sql.functions import max, datediff, lit

reference_date = df_purchases.select(max("InvoiceDate")).first()[0]

customer_recency = (
    df_purchases
    .groupBy("CustomerID")
    .agg(
        max("InvoiceDate").alias("last_purchase_date")
    )
    .withColumn(
        "recency_days",
        datediff(
            lit(reference_date),
            "last_purchase_date"
        )
    )
)

In [0]:
from pyspark.sql.functions import datediff, col, min, max

customer_tenure = (
    df_purchases
    .groupBy("CustomerID")
    .agg(
        min("InvoiceDate").alias("first_purchase_date"),
        max("InvoiceDate").alias("last_purchase_date")
    )
    .withColumn(
        "customer_tenure_days",
        datediff(
            col("last_purchase_date"),
            col("first_purchase_date") 
        )
    )
)

In [0]:
from pyspark.sql.functions import when, col

customer_returns =  (
    obs_df
    .groupBy("CustomerID")
    .agg(
        countDistinct("Invoice").alias("purchase_frequency"),
        countDistinct(
            when(col("is_cancellation"), col("Invoice"))
        ).alias("cancellation_count")
    )
)
customer_returns = customer_returns.withColumn(
    "return_rate",
    col("cancellation_count") /
    col("purchase_frequency")
)

In [0]:
from pyspark.sql.functions import lit, when, col

# Compare CustomerID in holdout and observed
active_in_holdout = holdout_purchases.select("CustomerID").distinct().withColumn("is_active", lit(1))
obs_customers = df_purchases.select("CustomerID").distinct()

# Assign churn_label: 1 if not active , 0 if active
churn_labels = (
    obs_customers
    .join(active_in_holdout, "CustomerID", "left")
    .withColumn("churn_label", when(col("is_active").isNull(), 1).otherwise(0))
    .select("CustomerID", "churn_label")
)

In [0]:
df_gold = (
    customer_order_features
    .join(customer_recency.select("CustomerID", "recency_days"), "CustomerID", "left")
    .join(customer_tenure.select("CustomerID", "customer_tenure_days"), "CustomerID", "left")
    .join(customer_returns.select("CustomerID", "return_rate"), "CustomerID", "left")
    .join(churn_labels, "CustomerID", "left")
)
df_gold.show()

+----------+-------------------+------------------+------------------+------------+--------------------+-------------------+-----------+
|CustomerID|average_order_value|       total_spend|purchase_frequency|recency_days|customer_tenure_days|        return_rate|churn_label|
+----------+-------------------+------------------+------------------+------------+--------------------+-------------------+-----------+
|     13578|             147.66|            147.66|                 1|         318|                   0|                0.0|          1|
|     17739|  429.2864285714286|           6010.01|                14|         105|                 536|              0.125|          0|
|     14239| 198.06368421052633|3763.2100000000005|                19|           3|                 640|0.13636363636363635|          0|
|     18200| 150.35000000000002|150.35000000000002|                 1|          15|                   0|                0.0|          1|
|     12505|  955.1999999999999| 955.1999

In [0]:
df_gold.count()

5281

In [0]:
df_gold.groupBy("CustomerID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+----------+-----+
|CustomerID|count|
+----------+-----+
+----------+-----+



In [0]:
gold_path = "s3://my-datasets-for-demo/customer-churn-datasets/gold/customer_features"

(
    df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_path)
)

In [0]:
df_gold_test = (
    spark.read
    .format("delta")
    .load(gold_path)
)

df_gold_test.show(5)
df_gold_test.printSchema()

+----------+-------------------+------------------+------------------+------------+--------------------+-------------------+-----------+
|CustomerID|average_order_value|       total_spend|purchase_frequency|recency_days|customer_tenure_days|        return_rate|churn_label|
+----------+-------------------+------------------+------------------+------------+--------------------+-------------------+-----------+
|     13578|             147.66|            147.66|                 1|         318|                   0|                0.0|          1|
|     17739|  429.2864285714286|           6010.01|                14|         105|                 536|              0.125|          0|
|     14239| 198.06368421052633|3763.2100000000005|                19|           3|                 640|0.13636363636363635|          0|
|     18200| 150.35000000000002|150.35000000000002|                 1|          15|                   0|                0.0|          1|
|     12505|  955.1999999999999| 955.1999